In [61]:
### NAIVE VERSION

In [62]:
cell_str='''
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

// --- Kernel Naive ---
__global__ void matMulNaive(const float * a, const float * b, float * c, int n) {

    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < n && col < n) {
        float sum = 0.0f;
        //#pragma unroll
        for (int k = 0; k < n; ++k) {
            sum += a[row * n + k] * b[k * n + col];
        }
        c[row * n + col] = sum;
    }
}

int main(int argc, char **argv) {
    if (argc < 2) return 1;
    int n = atoi(argv[1]);
    if (n <= 0) return 1;

    // CPU Memory Allocation
    size_t bytes = (size_t)n * n * sizeof(float);
    float *h_a = (float*)malloc(bytes);
    float *h_b = (float*)malloc(bytes);
    float *h_c = (float*)malloc(bytes);

    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            h_a[i * n + j] = 2.0f;
            h_b[i * n + j] = 3.0f;
            h_c[i * n + j] = 0.0f;
        }
    }

    // GPU Memory Allocation
    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    // Copy data from host to device
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // Grid and block dimensions
    int blockSize = 32;
    dim3 threadsPerBlock(blockSize, blockSize);
    dim3 numBlocks((n + blockSize - 1) / blockSize, (n + blockSize - 1) / blockSize);

    // CUDA Event for measuring execution time
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Launch of the kernel + timing
    cudaEventRecord(start);
    matMulNaive<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, n);
    cudaEventRecord(stop);

    // Wait for the kernel to finish
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("Execution Time: %.4f seconds\\n", milliseconds / 1000.0f);

    // Copy data from device to host
    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    FILE *f = fopen("mat-res.txt", "w");
    if (f) {
        fprintf(f, "%d\\n\\n", n);
        int sample_size = (n < 1000) ? n : 1000;
        for (int i = 0; i < sample_size; i++) {
            for (int j = 0; j < sample_size; j++) {
                fprintf(f, "%.0f ", h_c[i * n + j]);
            }
            fprintf(f, "\\n");
        }
        fclose(f);
    }

    // Deallocation: both CPU and GPU
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c);

    return 0;
}
'''

with open('cuda_naive.cu', 'w') as f:
    f.write(cell_str)

In [63]:
### OPTIMIZED VERSION

In [64]:
cell_str = r'''
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

// Configuration
// - BM, BN: size of the tile, work for each Thread Block
// - BK: step size, how many elements are loaded together for the multiplication
// - TM, TN: size of the block computed by each single thread
#define BM 128
#define BN 128
#define BK 32
#define TM 8
#define TN 8

// ---- KERNEL CUDA: 2D Register Tiling ----
__global__ void CudaKernel(int n, const float *A, const float *B, float *C) {
    // Shared Memory for the Tiles
    __shared__ float sA[BM][BK];
    __shared__ float sB[BK][BN];

    int bx = blockIdx.x;
    int by = blockIdx.y;
    int tx = threadIdx.x;
    int ty = threadIdx.y;

    // Starting index for the block 8x8 of this thread
    int row = by * BM + ty * TM;
    int col = bx * BN + tx * TN;

    // Registers to store the partial results of the multiplication
    float res[TM][TN] = {0.0f};

    // Linear index of the thread for Decoupled Loading
    int tid = ty * (BN / TN) + tx;

    // Loop
    for (int p = 0; p < (n + BK - 1) / BK; ++p) {
        // Decoupled Loading: 256 threads loads the data from the Global to the Shared

        // Loading of the Tile A
        for (int i = 0; i < (BM * BK) / 256; ++i) {
            int idx = i * 256 + tid;
            int a_row = idx / BK;
            int a_col = idx % BK;
            int g_a_row = by * BM + a_row;
            int g_a_col = p * BK + a_col;
            sA[a_row][a_col] = (g_a_row < n && g_a_col < n) ? A[g_a_row * n + g_a_col] : 0.0f;
        }

        // Loading of the Tile B
        for (int i = 0; i < (BK * BN) / 256; ++i) {
            int idx = i * 256 + tid;
            int b_row = idx / BN;
            int b_col = idx % BN;
            int g_b_row = p * BK + b_row;
            int g_b_col = bx * BN + b_col;
            sB[b_row][b_col] = (g_b_row < n && g_b_col < n) ? B[g_b_row * n + g_b_col] : 0.0f;
        }

        // Synchronization Barrier: ensures that all threads have finished loading the data before to start the computation
        __syncthreads();

        // Computation in the Registers
        #pragma unroll
        for (int k = 0; k < BK; ++k) {

            float a_reg[TM];
            float b_reg[TN];

            for(int i=0; i<TM; i++) a_reg[i] = sA[ty * TM + i][k];
            for(int j=0; j<TN; j++) b_reg[j] = sB[k][tx * TN + j];

            for(int i=0; i<TM; i++) {
                for(int j=0; j<TN; j++) {
                    res[i][j] += a_reg[i] * b_reg[j];
                }
            }
        }
        __syncthreads();
    }

    // Saving in the Global Memory
    for (int i = 0; i < TM; ++i) {
        for (int j = 0; j < TN; ++j) {
            int c_row = row + i;
            int c_col = col + j;
            if (c_row < n && c_col < n) {
                C[c_row * n + c_col] = res[i][j];
            }
        }
    }
}


// ---- HOST ----
int main(int argc, char **argv) {
    if (argc < 2) return 1;
    int n = atoi(argv[1]);
    if (n <= 0) return 1;

    // CPU Memory Allocation - Flattening 1D: ensures that the memory is contiguous
    size_t bytes = n * n * sizeof(float);
    float *h_a = (float*)malloc(bytes);
    float *h_b = (float*)malloc(bytes);
    float *h_c = (float*)malloc(bytes);

    // Initialization
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            h_a[i * n + j] = 2.0f;
            h_b[i * n + j] = 3.0f;
            h_c[i * n + j] = 0.0f;
        }
    }

    // GPU Memory Allocation & Transfering
    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    // Copy from CPU to GPU: bottleneck
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // Definition of Grid and Block for the execution - Ceiling approach
    dim3 threadsPerBlock(BN / TN, BM / TM);
    dim3 blocksPerGrid((n + BN - 1) / BN, (n + BM - 1) / BM);

    // Variables for measuring the execution time
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Kernel Launch + Time Measurement
    cudaEventRecord(start);
    CudaKernel<<<blocksPerGrid, threadsPerBlock>>>(n, d_a, d_b, d_c);
    cudaEventRecord(stop);

    // Synchronization: CPU waits for the GPU to finish
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("Fast Execution Time: %.4f seconds\n", milliseconds / 1000.0f);

    // Copy from GPU to CPU (RAM)
    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    FILE *f = fopen("mat-res-fast.txt", "w");
    if (f) {
        fprintf(f, "%d\n\n", n);
        int limit = (n < 1000) ? n : 1000;
        for (int i = 0; i < limit; i++) {
            for (int j = 0; j < limit; j++) {
                fprintf(f, "%.0f ", h_c[i * n + j]);
            }
            fprintf(f, "\n");
        }
        fclose(f);
    }

    // Memory deallocation of both CPU and GPU
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c);

    return 0;
}
'''

with open('cuda_opt.cu', 'w') as f:
    f.write(cell_str)

In [65]:
### LIBRARY

In [66]:
cell_str=r'''
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>
#include <cublas_v2.h>

int main(int argc, char **argv) {
    if (argc < 2) {
        fprintf(stderr, "Uso: %s \n", argv[0]);
        return 1;
    }

    int n = atoi(argv[1]);
    size_t bytes = n * n * sizeof(float);

    float *h_a = (float*)malloc(bytes);
    float *h_b = (float*)malloc(bytes);
    float *h_c = (float*)malloc(bytes);

    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            h_a[i * n + j] = 2.0f;
            h_b[i * n + j] = 3.0f;
            h_c[i * n + j] = 0.0f;
        }
    }

    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // ---- CuBLAS Logic ----

    cublasHandle_t handle;
    cublasCreate(&handle);

    // As for the previous library implentations, we set alpha and beta respectively to 1.0 and 0.0 in order to compute C = A * B
    float alpha = 1.0f;
    float beta = 0.0f;

    // CUDA Timing events
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    // Launch of the function "Sgemm" (Single precision GEneral Matrix Multiply)
    cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N,
                n, n, n,
                &alpha,
                d_b, n,
                d_a, n,
                &beta,
                d_c, n);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("cuBLAS Time for n=%d: %.4f seconds\n", n, milliseconds / 1000.0f);

    cublasDestroy(handle);

    // ---- End ----


    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    // Stampa di controllo
    FILE *f = fopen("mat-res.txt", "w");
    if (f) {
        fprintf(f, "%d\n\n", n);
        int sample = (n < 100) ? n : 10;
        for (int i = 0; i < sample; i++) {
            for (int j = 0; j < sample; j++) {
                fprintf(f, "%.0f ", h_c[i * n + j]);
            }
            fprintf(f, "\n");
        }
        fclose(f);
    }

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);

    return 0;
}
'''

with open('cuda_matrixmult_cuBLAS.cu', 'w') as f:
    f.write(cell_str)

In [67]:
### ADVANCED

In [1]:
cell_str = r'''
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define BM 128
#define BN 128
#define BK 16
#define TM 8
#define TN 8

// ---- KERNEL CUDA ----
__global__ void sgemm_float4_only(int pad_n, const float *A, const float *B, float *C) {

    __shared__ float sA[BM][BK];
    __shared__ float sB[BK][BN];

    int bx = blockIdx.x;
    int by = blockIdx.y;
    int tx = threadIdx.x;
    int ty = threadIdx.y;

    int tid = ty * (BN / TN) + tx; // Da 0 a 255 (256 thread in totale)

    float res[TM][TN] = {0.0f};
    float a_reg[TM];
    float b_reg[TN];

    for (int p = 0; p < (pad_n + BK - 1) / BK; ++p) {

        // Loading on Shared Memory through float4
        for (int i = 0; i < (BM * BK) / (256 * 4); ++i) {
            int load_idx = i * 256 + tid;
            int a_row = load_idx / (BK / 4);
            int a_col_v = (load_idx % (BK / 4)) * 4;

            int g_a_row = by * BM + a_row;
            int g_a_col = p * BK + a_col_v;

            float4 tmpA = make_float4(0.0f, 0.0f, 0.0f, 0.0f);
            if (g_a_row < pad_n && g_a_col < pad_n) {
                tmpA = *(const float4*)&A[g_a_row * pad_n + g_a_col];
            }
            *(float4*)&sA[a_row][a_col_v] = tmpA;
        }

        // Loading on Shared Memory through float4
        for (int i = 0; i < (BK * BN) / (256 * 4); ++i) {
            int load_idx = i * 256 + tid;
            int b_row = load_idx / (BN / 4);
            int b_col_v = (load_idx % (BN / 4)) * 4;

            int g_b_row = p * BK + b_row;
            int g_b_col = bx * BN + b_col_v;

            float4 tmpB = make_float4(0.0f, 0.0f, 0.0f, 0.0f);
            if (g_b_row < pad_n && g_b_col < pad_n) {
                tmpB = *(const float4*)&B[g_b_row * pad_n + g_b_col];
            }
            *(float4*)&sB[b_row][b_col_v] = tmpB;
        }

        __syncthreads();

        // Computation on Shared Memory and Registers
        #pragma unroll
        for (int k = 0; k < BK; ++k) {
            for(int i=0; i<TM; i++) a_reg[i] = sA[ty * TM + i][k];
            for(int j=0; j<TN; j++) b_reg[j] = sB[k][tx * TN + j];

            for(int i=0; i<TM; i++) {
                for(int j=0; j<TN; j++) {
                    res[i][j] += a_reg[i] * b_reg[j];
                }
            }
        }
        __syncthreads();
    }

    // Write in Global Memory
    int row_start = by * BM + ty * TM;
    int col_start = bx * BN + tx * TN;

    for (int i = 0; i < TM; ++i) {
        for (int j = 0; j < TN; ++j) {
            int c_row = row_start + i;
            int c_col = col_start + j;
            if (c_row < pad_n && c_col < pad_n) {
                C[c_row * pad_n + c_col] = res[i][j];
            }
        }
    }
}


// ---- HOST ----
int main(int argc, char **argv) {

    if (argc < 2) return 1;
    int n = atoi(argv[1]);

    // Arrounded for padding
    int pad_n = (n + BM - 1) / BM * BM;

    // Memory Allocation
    size_t pad_bytes = (size_t)pad_n * pad_n * sizeof(float);
    float *h_a = (float*)malloc(pad_bytes);
    float *h_b = (float*)malloc(pad_bytes);
    float *h_c = (float*)malloc(pad_bytes);

    // Padding loading
    for (int i = 0; i < pad_n; i++) {
        for (int j = 0; j < pad_n; j++) {
            if (i < n && j < n) {
                h_a[i * pad_n + j] = 2.0f;
                h_b[i * pad_n + j] = 3.0f;
            } else {
                h_a[i * pad_n + j] = 0.0f;
                h_b[i * pad_n + j] = 0.0f;
            }
            h_c[i * pad_n + j] = 0.0f;
        }
    }

    // CUDA Memory Allocation
    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, pad_bytes);
    cudaMalloc(&d_b, pad_bytes);
    cudaMalloc(&d_c, pad_bytes);

    // Data transferring
    cudaMemcpy(d_a, h_a, pad_bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, pad_bytes, cudaMemcpyHostToDevice);

    // Setting the Grid and Block
    dim3 threadsPerBlock(BN / TN, BM / TM);
    dim3 blocksPerGrid((pad_n + BN - 1) / BN, (pad_n + BM - 1) / BM);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Kernel Launch + Timing
    cudaEventRecord(start);
    sgemm_float4_only<<<blocksPerGrid, threadsPerBlock>>>(pad_n, d_a, d_b, d_c);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);

    printf("Execution Time: %.4f seconds\n", milliseconds / 1000.0f);

    // Copying from GPU to CPU
    cudaMemcpy(h_c, d_c, pad_bytes, cudaMemcpyDeviceToHost);

    // Saving
    FILE *f = fopen("mat-res-float4.txt", "w");
    if (f) {
        fprintf(f, "%d\n\n", n);
        int limit = (n < 1000) ? n : 1000;
        for (int i = 0; i < limit; i++) {
            for (int j = 0; j < limit; j++) {
                fprintf(f, "%.0f ", h_c[i * pad_n + j]);
            }
            fprintf(f, "\n");
        }
        fclose(f);
    }

    // Memory deallocation
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c);

    return 0;
}
'''

with open('cuda_float4.cu', 'w') as f:
    f.write(cell_str)

In [2]:
cell_str = r'''#!/bin/bash

# ==============================================================================
# GPU MATRIX MULTIPLICATION BENCHMARK
# ==============================================================================

# Parameters
SIZES=(10000 15000 20000)
NUM_RUNS=3

echo "====================================================================="
echo "Starting the compilation..."
echo "====================================================================="

# Compilations
nvcc -arch=sm_75 -O3 cuda_naive.cu -o run_naive
nvcc -arch=sm_75 -O3 cuda_opt.cu -o run_optimized
nvcc -arch=sm_75 -O3 cuda_matrixmult_cuBLAS.cu -o run_cublas -lcublas
nvcc -arch=sm_75 -O3 cuda_float4.cu -o run_float4

echo "Compilation Completed!"

# ==============================================================================
# Starting Execution...
# ==============================================================================

for N in "${SIZES[@]}"; do
    echo ""
    echo "#####################################################################"
    echo " Test: $N x $N Matrix"
    echo "#####################################################################"

    #echo ">>> Naive Version"
    #for ((i=1; i<=NUM_RUNS; i++)); do
    #    echo -n "  [Run $i/$NUM_RUNS] "
    #    ./run_naive $N
    #done
    #echo "---------------------------------------------------------------------"

    #echo ">>> Optimized Version"
    #for ((i=1; i<=NUM_RUNS; i++)); do
    #    echo -n "  [Run $i/$NUM_RUNS] "
    #    ./run_optimized $N
    #done
    #echo "---------------------------------------------------------------------"

    #echo ">>> Library version: cuBLAS"
    #for ((i=1; i<=NUM_RUNS; i++)); do
    #    echo -n "  [Run $i/$NUM_RUNS] "
    #    ./run_cublas $N
    #done
    #echo "---------------------------------------------------------------------"

    echo ">>> Advanced Version"
    for ((i=1; i<=NUM_RUNS; i++)); do
        echo -n "  [Run $i/$NUM_RUNS] "
        ./run_float4 $N
    done
    echo "#####################################################################"
done

echo ""
echo "====================================================================="
echo " Benchmark Completed!"
echo "====================================================================="
'''

with open('run_benchmark.sh', 'w') as f:
    f.write(cell_str)

In [3]:
!chmod +x run_benchmark.sh
!./run_benchmark.sh

Starting the compilation...
cc1plus: fatal error: cuda_naive.cu: No such file or directory
compilation terminated.
cc1plus: fatal error: cuda_opt.cu: No such file or directory
compilation terminated.
cc1plus: fatal error: cuda_matrixmult_cuBLAS.cu: No such file or directory
compilation terminated.
Compilation Completed!

#####################################################################
 Test: 10000 x 10000 Matrix
#####################################################################
>>> Advanced Version
  [Run 1/3] Execution Time: 0.5343 seconds
  [Run 2/3] Execution Time: 0.4452 seconds
  [Run 3/3] Execution Time: 0.4478 seconds
#####################################################################

#####################################################################
 Test: 15000 x 15000 Matrix
#####################################################################
>>> Advanced Version
  [Run 1/3] Execution Time: 1.5992 seconds
  [Run 2/3] Execution Time: 1.5308 seconds
  [Run 3/3] E